In [1]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import os
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00
Using device: cuda


In [2]:
# Download dataset
import kagglehub
path = kagglehub.dataset_download("braunge/aerial-view-car-detection-for-yolov5")
print(f"Dataset path: {path}")

# Let's explore the dataset structure
print("Exploring dataset structure...")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files
        if file.endswith(('.jpg', '.jpeg', '.png', '.txt')):
            print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files) - 5} more files')

# Let's find all image files in the dataset
def find_all_images(root_path):
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    image_paths = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.lower().endswith(image_extensions):
                image_paths.append(os.path.join(root, file))
    return image_paths

# Let's find all label files in the dataset
def find_all_labels(root_path):
    label_paths = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.lower().endswith('.txt'):
                label_paths.append(os.path.join(root, file))
    return label_paths

image_paths = find_all_images(path)
label_paths = find_all_labels(path)

print(f"Found {len(image_paths)} images and {len(label_paths)} label files")

# Analyze object sizes to determine thresholds
def analyze_sizes(label_paths, img_size=640):
    sizes = []
    for label_path in label_paths:
        try:
            with open(label_path, 'r') as f:
                for line in f:
                    data = line.strip().split()
                    if len(data) == 5:  # class, x_center, y_center, width, height
                        _, _, _, w, h = map(float, data)
                        sizes.append([w * img_size, h * img_size])
        except:
            continue

    if len(sizes) == 0:
        print("No valid label files found, using default thresholds")
        return 32, 32, 1024  # Default thresholds

    sizes = np.array(sizes)
    if len(sizes) > 2:
        kmeans = KMeans(n_clusters=2, random_state=42).fit(sizes)
        centers = kmeans.cluster_centers_
        small_idx = np.argmin(centers[:, 0] * centers[:, 1])
        thresholds = (centers[0] + centers[1]) / 2
    else:
        thresholds = [32, 32]  # Default thresholds

    return thresholds[0], thresholds[1], thresholds[0] * thresholds[1]

width_th, height_th, area_th = analyze_sizes(label_paths)
print(f"Thresholds - Width: {width_th:.1f}, Height: {height_th:.1f}, Area: {area_th:.1f}")

100%|██████████| 67.7M/67.7M [00:02<00:00, 23.7MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/braunge/aerial-view-car-detection-for-yolov5/versions/3
Exploring dataset structure...
3/
  mydata/
    mydata/
      labels/
        test/
          6 (17)_1649991717.txt
          5 (41)_1650424112.txt
          1 (18)_1649990925.txt
          5 (20)_1649991626.txt
          4 (47)_1650423582.txt
          ... and 14 more files
        train/
          4 (42)_1650424016.txt
          3 (46)_1650423880.txt
          2 (22)_1649991051.txt
          4 (11)_1649859966.txt
          3 (43)_1650424087.txt
          ... and 275 more files
      images/
        test/
          1 (4)_1649859559.jpg
          1 (18)_1649990925.jpg
          4 (13)_1649859983.jpg
          6 (26)_1649991767.jpg
          6 (17)_1649991717.jpg
          ... and 14 more files
        train/
          2 (49)_1650423572.jpg
          5 (48)_1650423547.jpg
          5 (12)_1649860110.jpg
          5 (42)_1650424039.jpg
          1 (24)_1649990952.jpg
          ... and 27

In [3]:
# Scene Analysis Module
class SceneAnalyzer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Simplified AFP-YOLO Model
class AFPYOLO(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.scene_analyzer = SceneAnalyzer()

        # Backbone (simplified)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
        )

        # Detection head
        self.head = nn.Sequential(
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
            nn.Conv2d(512, (5 + num_classes) * 3, 1)  # 3 anchors
        )

    def forward(self, x):
        scene_complexity = self.scene_analyzer(x)
        features = self.backbone(x)
        detection = self.head(features)
        return detection, scene_complexity

model = AFPYOLO().to(device)
print("Model created successfully!")

Model created successfully!


In [4]:
# Simple dataset that uses the found images
class CarDataset(Dataset):
    def __init__(self, image_paths, label_paths=None):
        self.image_paths = image_paths
        self.label_paths = label_paths or []

        # Create a mapping from image name to label path for easier lookup
        self.label_map = {}
        for label_path in self.label_paths:
            img_name = os.path.splitext(os.path.basename(label_path))[0]
            self.label_map[img_name] = label_path

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(img_path)

        if img is None:
            # Create a dummy image if file can't be read
            img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Try to find corresponding label
        img_name = os.path.splitext(os.path.basename(img_path))[0]
        label_path = self.label_map.get(img_name)

        # Count small vs large objects if label exists
        small_objects, large_objects = 0, 0
        if label_path and os.path.exists(label_path):
            try:
                with open(label_path, 'r') as f:
                    for line in f:
                        data = line.strip().split()
                        if len(data) == 5:  # class, x_center, y_center, width, height
                            _, _, _, w, h = map(float, data)
                            width_px = w * 640
                            height_px = h * 640
                            area_px = width_px * height_px

                            if (width_px < width_th and height_px < height_th and area_px < area_th):
                                small_objects += 1
                            else:
                                large_objects += 1
            except:
                pass

        # Calculate scene complexity
        total_objects = small_objects + large_objects
        scene_complexity = small_objects / total_objects if total_objects > 0 else 0.0

        img = cv2.resize(img, (640, 640))
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        # Return image, dummy target, and scene complexity
        return img_tensor, torch.zeros(1), scene_complexity

# Create dataset and dataloader
dataset = CarDataset(image_paths, label_paths)
print(f"Dataset size: {len(dataset)}")

if len(dataset) > 0:
    dataloader = DataLoader(dataset, batch_size=min(4, len(dataset)), shuffle=True)

    # Training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()  # Simplified loss

    # Training loop
    print("Starting training...")
    for epoch in range(10):  # Short training for demo
        total_loss = 0
        scene_complexities = []

        for images, targets, scene_comp in dataloader:
            images = images.to(device)

            optimizer.zero_grad()
            outputs, pred_scene_complexity = model(images)

            # Compare predicted scene complexity with actual
            scene_loss = criterion(pred_scene_complexity, scene_comp.to(device).view(-1, 1).float())

            # Add detection loss (simplified)
            detection_loss = criterion(outputs, torch.zeros_like(outputs))

            # Combined loss
            loss = scene_loss + detection_loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            scene_complexities.extend(scene_comp.numpy())

        avg_scene_complexity = np.mean(scene_complexities) if scene_complexities else 0
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}, Avg Scene Complexity: {avg_scene_complexity:.3f}")

    print("Training completed!")
else:
    print("No images found for training")

Dataset size: 299
Starting training...
Epoch 1, Loss: 0.0727, Avg Scene Complexity: 0.120
Epoch 2, Loss: 0.0552, Avg Scene Complexity: 0.120
Epoch 3, Loss: 0.0538, Avg Scene Complexity: 0.120
Epoch 4, Loss: 0.0537, Avg Scene Complexity: 0.120
Epoch 5, Loss: 0.0524, Avg Scene Complexity: 0.120
Epoch 6, Loss: 0.0531, Avg Scene Complexity: 0.120
Epoch 7, Loss: 0.0525, Avg Scene Complexity: 0.120
Epoch 8, Loss: 0.0524, Avg Scene Complexity: 0.120
Epoch 9, Loss: 0.0522, Avg Scene Complexity: 0.120
Epoch 10, Loss: 0.0527, Avg Scene Complexity: 0.120
Training completed!


In [5]:
# Test on sample images
def test_model(model, image_paths):
    if not image_paths:
        print("No images to test with")
        return

    # Test on first few images
    for i, img_path in enumerate(image_paths[:30]):
        img = cv2.imread(img_path)
        if img is None:
            print(f"Could not load image {img_path}, skipping")
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        display_img = img.copy()
        img = cv2.resize(img, (640, 640))
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0) / 255.0

        model.eval()
        with torch.no_grad():
            detection, scene_complexity = model(img_tensor.to(device))

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(display_img)
        plt.title("Original Image")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(img_tensor[0].cpu().permute(1, 2, 0))
        plt.title(f"Scene Complexity: {scene_complexity.item():.3f}")
        plt.axis('off')

        plt.show()

        if scene_complexity.item() > 0.5:
            print(f"Image {i+1}: Small objects detected - Using HR pathway")
        else:
            print(f"Image {i+1}: Large objects detected - Using standard pathway")

# Test on available images
test_model(model, image_paths)

# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'width_threshold': width_th,
    'height_threshold': height_th,
    'area_threshold': area_th
}, 'afp_yolo_model.pth')

print("Model saved as 'afp_yolo_model.pth'")
print("Implementation completed successfully!")

Output hidden; open in https://colab.research.google.com to view.